In [ ]:
# Canonical analysis definitions (single source of truth: analysis/_canonical_defs.py).
# Importing these overrides any local fallback definitions later in the notebook.
# Edits to filters / score-mapping / bucket-rule belong in _canonical_defs.py, not here.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("analysis").resolve()))
from _canonical_defs import (
    ANCHOR, L2_CELLS, AXIS_CELLS, VERDICT_SCORE,
    candidate_score, sign,
    any_tier_disagreement, boundary_flip, strict_reversal,
    same_side_magnitude_shift, strong_to_weak_shift,
    assign_bucket,
)


# 10e — AH anchor-swap robustness

*Renamed from `06e_anchor_swap.ipynb` 2026-05-25.*

**Reviewer concern:** the per-axis L3 ranking (F > L1 > C > D) might be an artefact of using gpt-4.1 with the default Arena-Hard rubric (cell C1F1D1) as anchor. Does the ranking hold under a different defensible anchor?

**Approach (no API calls):** re-pivot the existing 8-cell factorial. Under an alternative anchor, the multi-axis cells from the original frame become single-axis swaps from the new anchor.

**Alternative anchor:** `C_priority` (cell `C2F1D1` — multi-criteria priority rubric, reasoned verdict, no scope, gpt-4.1).

Under this alt anchor, single-axis swaps map to:
- **C-axis** = revert C2→C1 = `C1F1D1` (original anchor cell)
- **F-axis** = swap F1→F2 = `C2F2D1` (= CF_priority_label under original anchor)
- **D-axis** = swap D1→D2 = `C2F1D2` (= CD_priority_coding under original anchor)
- **L1-axis** = claude judge on C2F1D1 — *not run; reported as a limitation*

**Disagreement metric:** matches `ah_master_ladder.csv` from `10_analysis.ipynb`: `mean_score_anchor != mean_score_variant` over all valid prompts. Includes tie-vs-non-tie; does NOT restrict to sign-flips. Sanity check: this should reproduce the published §5.2 numbers (e.g., D_coding_scope at 0.2218 against the original anchor).

**Outputs (saved to `results/anchor_swap/`):**
1. `per_cell_disagreement_<anchor>.csv` — cell-vs-anchor disagreement under each anchor choice
2. `per_axis_l3_<anchor>.csv` — per-axis L3 (single cell per axis, so union = mean)
3. `ranking_comparison.csv` — axis ranking under original vs alternative anchor

## 1. Setup and data loading

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

AH = Path.cwd()
L2_DIR = AH / 'data' / 'full_factorial'
L1_DIR = AH / 'data' / 'L1_swap'
OUT = AH / 'results' / 'anchor_swap'
OUT.mkdir(parents=True, exist_ok=True)

L2_CELLS = ['C1F1D1', 'C1F1D2', 'C1F2D1', 'C1F2D2', 'C2F1D1', 'C2F1D2', 'C2F2D1', 'C2F2D2']
VERDICT_SCORE = {'A>>B': -2, 'A>B': -1, 'A=B': 0, 'B>A': 1, 'B>>A': 2}

def candidate_score(verdict, round_name):
    if verdict is None or verdict not in VERDICT_SCORE:
        return None
    base = VERDICT_SCORE[verdict]
    return base if round_name == 'round1' else -base

def load_cell(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

l2_data = {cid: load_cell(L2_DIR / f'{cid}.jsonl') for cid in L2_CELLS}
l1_path = L1_DIR / 'C1F1D1.jsonl'
l1_data = load_cell(l1_path) if l1_path.exists() else None
print(f'L2 factorial: {len(L2_CELLS)} cells × {len(l2_data["C1F1D1"])} prompts')
print(f'L1 swap (claude on C1F1D1): {len(l1_data) if l1_data else 0} prompts')

L2 factorial: 8 cells × 500 prompts
L1 swap (claude on C1F1D1): 498 prompts


In [2]:
def aggregate_to_score_df(cell_records, cell_id):
    rows = []
    for r in cell_records:
        s1 = candidate_score(r['round1']['verdict'], 'round1')
        s2 = candidate_score(r['round2']['verdict'], 'round2')
        mean = (s1 + s2) / 2 if s1 is not None and s2 is not None else None
        rows.append({'uid': r['uid'], 'cell_id': cell_id, 'mean_score': mean})
    return pd.DataFrame(rows)

all_rows = [aggregate_to_score_df(records, cid) for cid, records in l2_data.items()]
if l1_data:
    all_rows.append(aggregate_to_score_df(l1_data, 'L1_claude'))

results_df = pd.concat(all_rows, ignore_index=True)
wide = results_df.pivot(index='uid', columns='cell_id', values='mean_score')
print(f'Wide table: {wide.shape[0]} prompts × {wide.shape[1]} cells')
print(f'Columns: {list(wide.columns)}')

Wide table: 500 prompts × 9 cells
Columns: ['C1F1D1', 'C1F1D2', 'C1F2D1', 'C1F2D2', 'C2F1D1', 'C2F1D2', 'C2F2D1', 'C2F2D2', 'L1_claude']


## 2. Disagreement metric

Match the metric used in `unified_analysis.ipynb` to build `level1_aggregate.csv` (the source of the §5.2 numbers):

- **disagreement** = `sign(mean_score_anchor) != sign(mean_score_variant)`, where `sign` returns -1 (A wins), 0 (tie), +1 (B wins). Tie-vs-non-tie counts as disagreement.
- **small disagreement** = sign change involving a tie (one side is 0, the other is non-zero) — strength/tie-status change without direction reversal
- **big disagreement** = both non-zero with opposite signs — verdict-direction reversal

Cohen's κ is computed on the sign category as a string label.

Sanity check on the original anchor C1F1D1: D_coding_scope should be 0.2218, F_label_only 0.4194, C_priority 0.2601, L1_claude 0.3502 (matches level1_aggregate.csv).

In [3]:
from sklearn.metrics import cohen_kappa_score

def cell_disagreement(wide_df, anchor_cell, variant_cell):
    """AH disagreement metric (matches unified_analysis.ipynb / level1_aggregate.csv).

    Disagreement = sign(mean_score_anchor) != sign(mean_score_variant), where sign
    yields -1 (A wins), 0 (tie), +1 (B wins). Tie-vs-non-tie counts as disagreement.
    Cohen's kappa is computed on the sign category as a string label.
    """
    paired = wide_df[[anchor_cell, variant_cell]].dropna()
    a = paired[anchor_cell].values
    b = paired[variant_cell].values
    n = len(a)
    if n == 0:
        return None

    a_sign = np.sign(a).astype(int)
    b_sign = np.sign(b).astype(int)

    n_disagree = int((a_sign != b_sign).sum())
    # severity breakdown on sign comparison:
    #   small = sign change involving a tie (one is 0, the other is non-zero)
    #   big   = sign change between A-favoured and B-favoured (both non-zero, opposite signs)
    one_zero = (a_sign == 0) | (b_sign == 0)
    n_small = int(((a_sign != b_sign) & one_zero).sum())
    n_big = int(((a_sign != b_sign) & ~one_zero).sum())

    a_cat = a_sign.astype(str)
    b_cat = b_sign.astype(str)
    kappa = cohen_kappa_score(a_cat, b_cat)

    return {
        'n_paired': n,
        'n_disagreements': n_disagree,
        'n_small': n_small,
        'n_big': n_big,
        'agreement_rate': round(1 - n_disagree / n, 4),
        'disagreement_rate': round(n_disagree / n, 4),
        'small_rate': round(n_small / n, 4),
        'big_rate': round(n_big / n, 4),
        'cohens_kappa': round(kappa, 4),
    }

## 3. Anchor-mappings

Define which cell tests which axis under each anchor choice.

In [4]:
ANCHOR_MAPPINGS = {
    # Original anchor: C1F1D1 (default rubric, reasoned, no scope, gpt-4.1)
    'C1F1D1': {
        'C':  ('C2F1D1', 'C_priority'),
        'F':  ('C1F2D1', 'F_label_only'),
        'D':  ('C1F1D2', 'D_coding_scope'),
        'L1': ('L1_claude', 'L1_claude'),
    },
    # Alternative anchor: C2F1D1 (priority rubric, reasoned, no scope, gpt-4.1 = C_priority)
    'C2F1D1': {
        'C':  ('C1F1D1', 'C_default_revert'),
        'F':  ('C2F2D1', 'F_label_only'),    # = CF_priority_label under original anchor
        'D':  ('C2F1D2', 'D_coding_scope'),  # = CD_priority_coding under original anchor
        # L1 axis not testable: would need claude judge on C2F1D1
    },
}

ANCHOR_LABELS = {'C1F1D1': 'anchor (default)', 'C2F1D1': 'C_priority'}

def disagreement_table(wide_df, anchor_cell):
    rows = []
    for axis, (variant_cell, variant_label) in ANCHOR_MAPPINGS[anchor_cell].items():
        if variant_cell not in wide_df.columns:
            print(f'  skipping {axis}: cell {variant_cell} not in data')
            continue
        d = cell_disagreement(wide_df, anchor_cell, variant_cell)
        if d is None:
            continue
        rows.append({
            'anchor': ANCHOR_LABELS[anchor_cell],
            'anchor_cell': anchor_cell,
            'axis': axis,
            'variant_cell': variant_cell,
            'variant_label': variant_label,
            **d,
        })
    return pd.DataFrame(rows)

disagreement = {a: disagreement_table(wide, a) for a in ANCHOR_MAPPINGS}
for a, df in disagreement.items():
    print(f'\n=== anchor = {ANCHOR_LABELS[a]} ({a}) ===')
    print(df.sort_values('disagreement_rate').to_string(index=False))
    df.to_csv(OUT / f'per_cell_disagreement_{a}.csv', index=False)


=== anchor = anchor (default) (C1F1D1) ===
          anchor anchor_cell axis variant_cell  variant_label  n_paired  n_disagreements  n_small  n_big  agreement_rate  disagreement_rate  small_rate  big_rate  cohens_kappa
anchor (default)      C1F1D1    D       C1F1D2 D_coding_scope       496              110      101      9          0.7782             0.2218      0.2036    0.0181        0.6399
anchor (default)      C1F1D1    C       C2F1D1     C_priority       496              129      107     22          0.7399             0.2601      0.2157    0.0444        0.5838
anchor (default)      C1F1D1   L1    L1_claude      L1_claude       494              173      131     42          0.6498             0.3502      0.2652    0.0850        0.4450
anchor (default)      C1F1D1    F       C1F2D1   F_label_only       496              208      190     18          0.5806             0.4194      0.3831    0.0363        0.3885

=== anchor = C_priority (C2F1D1) ===
    anchor anchor_cell axis variant_ce

## 4. Per-axis L3 (= single-cell disagreement, since AH has one variant per axis)

In [5]:
for a, df in disagreement.items():
    axis_l3 = df[['axis', 'disagreement_rate']].rename(
        columns={'disagreement_rate': 'l3_disagreement'}
    ).copy()
    axis_l3['anchor'] = ANCHOR_LABELS[a]
    axis_l3 = axis_l3[['anchor', 'axis', 'l3_disagreement']]
    print(f'\n=== L3 under anchor = {ANCHOR_LABELS[a]} ===')
    print(axis_l3.sort_values('l3_disagreement', ascending=False).to_string(index=False))
    axis_l3.to_csv(OUT / f'per_axis_l3_{a}.csv', index=False)


=== L3 under anchor = anchor (default) ===
          anchor axis  l3_disagreement
anchor (default)    F           0.4194
anchor (default)   L1           0.3502
anchor (default)    C           0.2601
anchor (default)    D           0.2218

=== L3 under anchor = C_priority ===
    anchor axis  l3_disagreement
C_priority    F           0.4640
C_priority    C           0.2601
C_priority    D           0.2100


## 5. Ranking comparison

In [6]:
rank_rows = []
for a, df in disagreement.items():
    ranked = df.sort_values('disagreement_rate', ascending=False)['axis'].tolist()
    rank_rows.append({
        'anchor': ANCHOR_LABELS[a],
        'anchor_cell': a,
        'axis_ranking': ' > '.join(ranked),
        'n_axes_tested': len(ranked),
    })
ranking = pd.DataFrame(rank_rows)
print(ranking.to_string(index=False))
ranking.to_csv(OUT / 'ranking_comparison.csv', index=False)

          anchor anchor_cell   axis_ranking  n_axes_tested
anchor (default)      C1F1D1 F > L1 > C > D              4
      C_priority      C2F1D1      F > C > D              3


## 6. Headline number for paper draft

Robustness claim for §6 / §7: under alternative anchor `C_priority`, the per-axis ranking is _____. F-top and D-bottom should hold; L1 axis not testable without a new claude run on C2F1D1 (limitation to note in main text).

If the alt-anchor ranking restricted to {C, F, D} preserves F > C > D from the original, the reviewer's anchor-induced-bias concern is materially closed for AH.

In [7]:
print('AH anchor-swap robustness summary:')
for a, df in disagreement.items():
    ranked = df.sort_values('disagreement_rate', ascending=False)
    rank_str = ' > '.join(
        f'{r.axis} ({r.disagreement_rate:.3f})' for _, r in ranked.iterrows()
    )
    print(f'  anchor = {ANCHOR_LABELS[a]:>20s}  ->  {rank_str}')

AH anchor-swap robustness summary:
  anchor =     anchor (default)  ->  F (0.419) > L1 (0.350) > C (0.260) > D (0.222)
  anchor =           C_priority  ->  F (0.464) > C (0.260) > D (0.210)
